In [186]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score,roc_auc_score
from collections import defaultdict

In [187]:
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.xls')

In [188]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [189]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [190]:
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())

In [191]:
df['HasFamily'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).map({True: 'Yes', False: 'No'})

In [192]:
#df['Is_streaming'] = ((df['StreamingMovies'] == 'Yes') | (df['StreamingTV'] == 'Yes')).map({True:'Yes',False:'No'})

df['Is_streaming'] = np.where(
    (df['StreamingMovies'] == 'Nointernetservice') | (df['StreamingTV'] == 'Nointernetservice'), 'Nointernetservice',
        np.where(
            (df['StreamingMovies'] == 'Yes') | (df['StreamingTV'] == 'Yes'), 'Yes', 'No'))

In [193]:
#df['Online_backup_security'] = ((df['OnlineBackup'] == 'Yes') | (df['OnlineSecurity'] == 'Yes')).map({True:"Yes",False:'No'})
df['Online_backup_security'] = np.where(
    (df['OnlineSecurity'] == 'Nointernetservice') | (df['OnlineBackup'] == 'Nointernetservice'), 'Nointernetservice',
        np.where(
            (df['OnlineSecurity'] == 'Yes') | (df['OnlineBackup'] == 'Yes'), 'Yes', 'No'))

In [194]:
drop_columns = ['customerID','OnlineBackup','OnlineSecurity','Partner','Dependents','StreamingMovies','StreamingTV']
for col in drop_columns:
    if col in df.columns:
        df.drop(columns=[col],inplace=True)
#df.drop(columns=['customerID','Partner','Dependents','StreamingMovies','StreamingTV'],inplace=True)
df.columns

Index(['gender', 'SeniorCitizen', 'tenure', 'PhoneService', 'MultipleLines',
       'InternetService', 'DeviceProtection', 'TechSupport', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges',
       'Churn', 'HasFamily', 'Is_streaming', 'Online_backup_security'],
      dtype='str')

In [195]:
X = df.drop(columns=['Churn'])
y = df['Churn']
X

,gender,SeniorCitizen,tenure,PhoneService,MultipleLines,InternetService,DeviceProtection,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,HasFamily,Is_streaming,Online_backup_security
0,Female,0,1,No,No phone service,DSL,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,Yes,No,Yes
1,Male,0,34,Yes,No,DSL,Yes,No,One year,No,Mailed check,56.95,1889.50,No,No,Yes
2,Male,0,2,Yes,No,DSL,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,No,No,Yes
3,Male,0,45,No,No phone service,DSL,Yes,Yes,One year,No,Bank transfer (automatic),42.30,1840.75,No,No,Yes
4,Female,0,2,Yes,No,Fiber optic,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,24,Yes,Yes,DSL,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,Yes,Yes,Yes
7039,Female,0,72,Yes,Yes,Fiber optic,Yes,No,One year,Yes,Credit card (automatic),103.20,7362.90,Yes,Yes,Yes
7040,Female,0,11,No,No phone service,DSL,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,Yes,No,Yes
7041,Male,1,4,Yes,Yes,Fiber optic,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes,No,No


In [196]:
numerical_feature = X.select_dtypes(exclude="str").columns
categorical_feature = X.select_dtypes(include="str").columns
categorical_feature

Index(['gender', 'PhoneService', 'MultipleLines', 'InternetService',
       'DeviceProtection', 'TechSupport', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'HasFamily', 'Is_streaming', 'Online_backup_security'],
      dtype='str')

In [197]:
for col in categorical_feature:
    df[col] =  df[col].str.replace(' ', '')
    df[col] =  df[col].str.replace('-', '')

In [198]:
X.head()

,gender,SeniorCitizen,tenure,PhoneService,MultipleLines,InternetService,DeviceProtection,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,HasFamily,Is_streaming,Online_backup_security
0,Female,0,1,No,No phone service,DSL,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,Yes,No,Yes
1,Male,0,34,Yes,No,DSL,Yes,No,One year,No,Mailed check,56.95,1889.50,No,No,Yes
2,Male,0,2,Yes,No,DSL,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,No,No,Yes
3,Male,0,45,No,No phone service,DSL,Yes,Yes,One year,No,Bank transfer (automatic),42.30,1840.75,No,No,Yes
4,Female,0,2,Yes,No,Fiber optic,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,No,No,No


In [199]:
preprocessor = ColumnTransformer([
    ('OHE',OneHotEncoder(drop='first'),categorical_feature),
    ('SC',StandardScaler(),numerical_feature)
])

In [200]:
le = LabelEncoder()

y = le.fit_transform(y)
y

array([0, 0, 1, ..., 0, 1, 0], shape=(7043,))

In [201]:
X = preprocessor.fit_transform(X)


In [202]:
X

array([[ 0.        ,  0.        ,  1.        , ..., -1.27744458,
        -1.16032292, -0.99497138],
       [ 1.        ,  1.        ,  0.        , ...,  0.06632742,
        -0.25962894, -0.17387565],
       [ 1.        ,  1.        ,  0.        , ..., -1.23672422,
        -0.36266036, -0.96039939],
       ...,
       [ 0.        ,  0.        ,  1.        , ..., -0.87024095,
        -1.1686319 , -0.85518222],
       [ 1.        ,  1.        ,  0.        , ..., -1.15528349,
         0.32033821, -0.87277729],
       [ 1.        ,  1.        ,  0.        , ...,  1.36937906,
         1.35896134,  2.01391739]], shape=(7043, 23))

In [203]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)
print(X_train)
print(y_train)

[[ 1.          1.          0.         ...  0.88073469  0.19736523
   0.65642602]
 [ 1.          1.          0.         ... -1.27744458  0.52473924
  -0.97258569]
 [ 1.          1.          0.         ... -0.78880022 -1.51096208
  -0.89350724]
 ...
 [ 1.          1.          0.         ... -0.82952058 -1.44947559
  -0.87302013]
 [ 1.          1.          0.         ... -0.82952058  1.15289851
  -0.47824601]
 [ 1.          1.          0.         ... -0.25943549 -1.49434411
  -0.80623836]]
[0 0 0 ... 0 1 0]


In [204]:
models = {
                "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
                "Random Forest": RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42),
                "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
                "LightGBM": LGBMClassifier(class_weight='balanced', random_state=42),
                "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
                "Gradient Boosting": GradientBoostingClassifier(random_state=42),
                "SVM": SVC(class_weight='balanced', probability=True, random_state=42),
                "KNN": KNeighborsClassifier(n_neighbors=5)
            }

In [205]:
if 'report' not in locals():
    report = defaultdict(list)


In [206]:
for model_name,model in models.items():

    model.fit(X_train,y_train)

    predict = model.predict(X_test)

    roc_auc = roc_auc_score(y_test,predict)*100
    f1__score = f1_score(y_test,predict)*100

    report[model_name].append((roc_auc, f1__score))

    #report[model_name] = roc_auc,f1__score

[LightGBM] [Info] Number of positive: 1295, number of negative: 3635
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000797 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 625
[LightGBM] [Info] Number of data points in the train set: 4930, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


c:\Users\Dhvanish\OneDrive\Desktop\ML projects\Customer churn prediction\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [207]:
report

defaultdict(list,
            {'Logistic Regression': [(78.16662251835551, 65.12890094979647),
              (78.04702587543837, 64.9932157394844),
              (78.41758868716506, 65.36124240378123),
              (78.13271887940266, 65.1639344262295),
              (77.8728098475638, 64.80978260869566),
              (77.8728098475638, 64.80978260869566)],
             'Random Forest': [(74.23023457469328, 62.00657894736842),
              (73.71794436407187, 61.359867330016584),
              (73.61012060413003, 61.17065127782357),
              (73.27346143135617, 60.69078947368421),
              (73.71794436407187, 61.359867330016584),
              (73.71794436407187, 61.359867330016584)],
             'XGBoost': [(70.10881992696284, 56.67627281460135),
              (69.924246026086, 56.375838926174495),
              (70.51045635769641, 57.33590733590733),
              (68.63692655305836, 54.31119920713577),
              (70.41440548072984, 57.14285714285714),
             

In [208]:

# for model, scores in sorted(report.items(), key=lambda item: item[1], reverse=True):
#     print(f"{model} -> roc_auc_score: {scores[0]:.4f}")
#     print(f"{model} -> f1_score: {scores[1]}")
#     print("\n")

table_data = []
for model_name, scores in report.items():
    # Fetch old and new scores safely
    old_roc, old_f1 = scores[0] if len(scores) >= 2 else (None, None)
    new_roc, new_f1 = scores[-1] if scores else (None, None)
    
    table_data.append({
        'Model Name': model_name,
        'Old ROC-AUC': old_roc,
        'New ROC-AUC': new_roc,
        'Old F1': old_f1,
        'New F1': new_f1
    })

# Convert to DataFrame and display
df_results = pd.DataFrame(table_data)
print(df_results)  # Or just type 'df_results' if in a Jupyter notebook cell


            Model Name  Old ROC-AUC  New ROC-AUC     Old F1     New F1
0  Logistic Regression    78.166623    77.872810  65.128901  64.809783
1        Random Forest    74.230235    73.717944  62.006579  61.359867
2              XGBoost    70.108820    70.414405  56.676273  57.142857
3             LightGBM    76.501665    76.454574  63.860668  64.000000
4             CatBoost    71.172002    70.943166  58.455523  58.096173
5    Gradient Boosting    71.725724    71.334445  59.390364  58.742633
6                  SVM    77.521548    78.097400  64.591978  65.280665
7                  KNN    69.181253    68.937814  55.141037  54.784240


In [209]:

# 1. Format the scores as strings "ROC / F1" for clean rows
formatted_report = {}
for model_name, scores in report.items():
    formatted_report[model_name] = [f"{roc:.4f} / {f1:.4f}" for roc, f1 in scores]

# 2. Create the DataFrame (this automatically sets model names as column headers)
# We use pd.DataFrame.from_dict with orient='columns' (default)
df_transposed = pd.DataFrame.from_dict(formatted_report, orient='columns')

# 3. Label the row index to represent the experiment runs
df_transposed.index = [f"Run {i+1}" for i in range(len(df_transposed))]

# Display the DataFrame
df_transposed


,Logistic Regression,Random Forest,XGBoost,LightGBM,CatBoost,Gradient Boosting,SVM,KNN
Run 1,78.1666 / 65.1289,74.2302 / 62.0066,70.1088 / 56.6763,76.5017 / 63.8607,71.1720 / 58.4555,71.7257 / 59.3904,77.5215 / 64.5920,69.1813 / 55.1410
Run 2,78.0470 / 64.9932,73.7179 / 61.3599,69.9242 / 56.3758,76.4781 / 63.9296,71.0953 / 58.3497,71.2252 / 58.5799,77.2174 / 64.2265,69.1916 / 55.1598
Run 3,78.4176 / 65.3612,73.6101 / 61.1707,70.5105 / 57.3359,75.3796 / 62.6010,71.1824 / 58.4887,71.2134 / 58.5899,78.1520 / 65.3287,69.5843 / 55.7466
Run 4,78.1327 / 65.1639,73.2735 / 60.6908,68.6369 / 54.3112,75.9997 / 63.3431,71.8025 / 59.4912,71.6711 / 59.3103,77.1628 / 64.1770,68.9199 / 54.7445
Run 5,77.8728 / 64.8098,73.7179 / 61.3599,70.4144 / 57.1429,76.4546 / 64.0000,70.9432 / 58.0962,71.3344 / 58.7426,78.0974 / 65.2807,68.9378 / 54.7842
Run 6,77.8728 / 64.8098,73.7179 / 61.3599,70.4144 / 57.1429,76.4546 / 64.0000,70.9432 / 58.0962,71.3344 / 58.7426,78.0974 / 65.2807,68.9378 / 54.7842
